# Lab 0 — L'ambiente di lavoro
Lab 0 serve a verificare se la vostra macchina è pronta per il resto dei lab.
## Prima di iniziare

**Teoria di riferimento**

- Blocco 0.1, «The publish/subscribe arrangement»: il nodo pubblica, un broker inoltra a chi si è sottoscritto.
- Blocco 0.1, «Topic names and wildcards»: i nomi dei topic e i filtri con `+` e `#`.
- Blocco 0.1, «QoS 0, 1 and 2 exchanges»: quali pacchetti si scambiano client e broker.

**Che cosa vediamo oggi**

1. Installare e testare il broker MQTT sul pc, con un sottoscrittore e un pubblicatore da terminale.
2. Lo stesso scambio da Python, con la libreria `paho-mqtt`, ricevuto sia dal terminale sia dal notebook.
3. Il repository del corso con Git: dove sono le cose e come si annulla una modifica.



### Preparazione: terminali

Si aprono tre terminali, tutti nella radice del repository e con l'ambiente virtuale
attivo (`source .venv/bin/activate` su MacOS/Linux; su Windows `.venv\Scripts\activate`).

**Terminale 1 — il broker.**

```
mosquitto -v
```

Avvia Mosquitto in primo piano sulla porta 1883; `-v` sta per *verbose*: stampa ogni pacchetto che il broker
riceve e invia. Senza file di configurazione, Mosquitto 2.x accetta connessioni solo da
`localhost`: per il corso basta. Se il comando risponde `Address already in use`, un broker
è già in esecuzione come servizio: va bene, si usa quello e il terminale 1 resta libero.

**Terminale 2 — un sottoscrittore.**

```
mosquitto_sub -h localhost -t 'test/#' -v
```

Si sottoscrive a tutti i topic che iniziano con `test/`; `-v` stampa il topic prima del
messaggio. Il comando non termina: resta in attesa.

**Terminale 3 — un pubblicatore.**

```
mosquitto_pub -h localhost -t test/ciao -m "ciao"
```

Nel terminale 2 deve comparire `test/ciao ciao`; nel terminale 1 il broker registra CONNECT,
PUBLISH e DISCONNECT del pubblicatore.

**Windows.** L'installatore di mosquitto.org mette i programmi in
`C:\Program Files\mosquitto\` e registra il servizio «Mosquitto Broker»; se il servizio
è avviato, la porta 1883 è già occupata e il broker c'è. La cartella non è nel `PATH`: o si aggiunge, o si lancia
`& "C:\Program Files\mosquitto\mosquitto_sub.exe" ...` da PowerShell. Per vedere il log
con `mosquitto -v` si ferma prima il servizio (`net stop mosquitto`, da amministratore). In
`cmd.exe` il filtro va fra virgolette doppie: `-t "test/#"`.

**macOS.** `brew install mosquitto`. Il broker sta in `$(brew --prefix)/sbin`
(`/opt/homebrew/sbin` su Apple Silicon, `/usr/local/sbin` su Intel), che può non essere nel
`PATH`; `brew services start mosquitto` lo avvia come servizio, con la configurazione in
`$(brew --prefix)/etc/mosquitto/mosquitto.conf`.

**Linux.** `sudo apt install mosquitto mosquitto-clients`: il servizio parte da solo
(`systemctl status mosquitto`), quindi `mosquitto -v` trova la porta occupata. O si usa il
servizio, o lo si ferma con `sudo systemctl stop mosquitto`.

### Chiudere il broker

Se è aperto in un terminale, basta `Ctrl+C` in quel terminale. Se gira in background o come servizio:

| sistema | comando |
|---|---|
| macOS | `brew services stop mosquitto`, oppure `pkill mosquitto` |
| Linux | `sudo systemctl stop mosquitto`, oppure `pkill mosquitto` |
| Windows | `net stop mosquitto` (da amministratore), oppure `taskkill /IM mosquitto.exe /F` |

Per vedere chi occupa la porta 1883: `lsof -i :1883` su macOS e Linux, `netstat -ano | findstr 1883` su Windows.

### La verifica

Nel terminale 3, dalla radice del repository:

```
python scripts/verify_setup.py --quick
```

lo script controlla i pacchetti installati, quale copia di `iomtlab` viene importata e il broker, poi
fa girare i test (un minuto circa). Scrive il rapporto in `verifica.txt`, nella radice:
se un controllo fallisce, è il file da mandare al docente.

### Setup del notebook

In [ ]:
# gli import
# una libreria o un modulo esterno si importano con il comando import seguito dal nome
import importlib
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt # as assegna un alias (un nome più corto) al modulo

from iomtlab.config import REPO_ROOT # from ... import importa solo alcune parti di un modulo o di una libreria
from iomtlab.net import broker_available, mqtt_client

WORK = REPO_ROOT / "work" # tutti gli output andranno nella cartella work
# REPO_ROOT è un oggetto Path di os (PosixPath su macOS/Linux, WindowsPath su Windows), con metodi per lavorare con cartelle e file
# l'operatore / unisce due pezzi di percorso, con il separatore giusto per il sistema
WORK.mkdir(exist_ok=True) # il metodo mkdir crea la cartella di lavoro; exist_ok=True: nessun errore se esiste già
print("repository:", REPO_ROOT)

## Parte 1 — Versioni e servizi

> **Domanda.** Le librerie e il broker ci sono? 

Il notebook usa le librerie installate nell'ambiente `.venv`, la copia di `iomtlab` di questo
repository e il broker sulla porta 1883. Controlliamo le tre cose in una tabella: se una riga dice `ASSENTE`,
il problema è nell'ambiente, non nel codice dei laboratori. Teoria: Blocco 0.1,
«The publish/subscribe arrangement» (il broker a cui il nodo pubblica) e «Scope, transport and
versions» (la porta 1883 e il trasporto TCP).

In [ ]:
PACCHETTI = ["numpy", "pandas", "scipy", "sklearn", "matplotlib", "torch", "paho.mqtt", "iomtlab"]
# PACCHETTI è una lista, la struttura dati base di Python
# le liste sono iterabili, quindi con un ciclo for si possono scorrere senza un indice dedicato
# la sintassi è for x in iterable:, in cui x è una variabile nella quale viene scritto l'elemento dell'iterabile a ogni ciclo

# sintassi per creare una funzione
# def nome(argomenti):
def controlla(host, port):
    righe = [("python", sys.version.split()[0])] # creo una lista di tuple (che cosa, stato); la prima è la versione di Python
    # sys.version è una stringa, split() la spezza sugli spazi, [0] prende il primo pezzo
    for nome in PACCHETTI:
        try: # try/except è il costrutto per il controllo degli errori
            # dopo il try ci sono gli statement di cui vogliamo "catturare" eventuali errori
            m = importlib.import_module(nome) # importa il modulo a partire dal suo nome scritto in una stringa
            righe.append((nome, getattr(m, "__version__", "?"))) # getattr legge l'attributo __version__; se non c'è, restituisce "?"
        except ImportError: # except viene esegito quando nel try si verifica un errore del tipo indicato (qui ImportError)
            righe.append((nome, "ASSENTE")) # istruzione da eseguire in caso di errore (rende l'errore non bloccante)
    import iomtlab
    da = Path(iomtlab.__file__).resolve().parent # la cartella da cui Python ha caricato iomtlab (reslve da il path assoluto)
    radice = REPO_ROOT.resolve() # resolve() trasforma il percorso in assoluto
    if da.is_relative_to(radice): # la libreria sta dentro il repository?
        righe.append(("iomtlab da", f"./{da.relative_to(radice).as_posix()}")) # f"...{...}": una f-string inserisce il valore fra graffe nel testo
    else:
        righe.append(("iomtlab da", f"{da} (fuori dal repository)")) # caso tipico: un'altra copia, per esempio installata da conda
    # testiamoil broker: proviamo ad aprire una connessione TCP sulla porta
    righe.append((f"broker {host}:{port}", "risponde" if broker_available(host, port) else "ASSENTE")) # a if condizione else b: sceglie fra due valori
    return pd.DataFrame(righe, columns=["che cosa", "stato"]).set_index("che cosa") # dalla lista di tuple a una tabella con due colonne
    # pd è l'alias di import di pandas, un alibreria per gestire dtaframe (simili a tabelle csv/excel)
controlla("localhost", 1883) # invocazione della funzione

### Prova a cambiare

Si cambiano solo i valori fra le due righe `PARAMETRI`, poi si riesegue la cella.

**Prima di rieseguire: che cosa ti aspetti?** Sulla porta 1884 non ascolta nessuno: che cosa scrive la riga del broker, e perché la connessione fallisce subito invece di attendere?

In [ ]:
# PARAMETRI 
BROKER_HOST = "localhost"   # dove cerco il broker: "localhost" o "127.0.0.1"
BROKER_PORT = 1883   # la porta: 1883 è quella di MQTT; 8883 è MQTT su TLS; prova 1884
# ───────────────────────────────────────────────────────────────────────────
controlla(BROKER_HOST, BROKER_PORT) # controlla è la funzione scritta sopra: la riga del broker dice se la porta accetta una connessione

**Che cosa guardare.** Solo la riga del broker cambia. La connessione a una porta chiusa di `localhost` viene rifiutata immediatamente dal sistema operativo (un segmento RST), quindi la cella non aspetta il timeout di un secondo.

1. Se il broker gira su un'altra macchina della rete, quale dei due parametri cambi, e che cosa deve permettere la configurazione di Mosquitto 2.x?
2. La riga del broker dice solo che la porta accetta una connessione TCP: basta per dire che dall'altra parte c'è un broker MQTT? Che cosa servirebbe per esserne certi?

## Parte 2 — Un messaggio da Python

> **Domanda.** Un messaggio mandato da Python arriva a chi lo aspetta? 

Stesso scambio dei terminali 2 e 3, ma da Python. Un client del notebook si sottoscrive con
un filtro e raccoglie i messaggi; un secondo client pubblica. Se il terminale 2 è ancora
aperto, il messaggio compare anche lì: il broker consegna a tutti i sottoscrittori il cui
filtro corrisponde al topic. Teoria: Blocco 0.1, «Topic names and wildcards».

In [ ]:
# paho-mqtt (una delle librerie istallate dalla repo) gestisce mqtt da Python, in maniera simile ai client da terminale
# ma consentendo di gestire tutit gli aspetti del clinet
arrivati = [] # lista in cui ci saranno i messaggi che arrivano
# un thread è la più piccola unità di elaborazione che può essere eseguita da un sistema operativo all'interno di un programma
# il main di un programma è il suo thread principale
# paho-mqtt avvia un secondo thread, quello di rete: è lui a ricevere i messaggi e a chiamare on_message, mentre il notebook va avanti

def on_message(client, userdata, msg): # funzione di callback: la chiama paho-mqtt quando arriva un messaggio
    arrivati.append({"topic": msg.topic, "testo": msg.payload.decode(),
                     "qos": msg.qos, "t": time.perf_counter()}) # aggiungo il messaggio alla lista, come dizionario; decode() trasforma i byte in testo
    # t è l'istante di arrivo, segnato qui dentro: il ritardo misurato non dipende da quanto aspetta il notebook

def iscriviti(topic, qos=0): # funzione di sottoscrizione che restituisce un client iscritto a un certo topic; qos=0 è il valore di default
    sub = mqtt_client("notebook-sub", 1883, on_message=on_message) # crea il client, si collega e avvia il thread di rete; qui si registra la callback
    sub.subscribe(topic, qos=qos)
    time.sleep(0.3) # lascio al broker il tempo di registrare la sottoscrizione prima di pubblicare
    return sub

def pubblica(topic, testo, qos=0): # funzione di pubblicazione
    pub = mqtt_client("notebook-pub", 1883) # prima crea il client
    t0 = time.perf_counter() # l'istante di partenza, in secondi, da un orologio adatto a misurare intervalli con precisione
    pub.publish(topic, testo, qos=qos).wait_for_publish() # invia il messaggio e aspetta che sia partito: a QoS 0 quando è uscito, a QoS 1 e 2 quando il broker l'ha confermato
    return pub, t0 # una funzione può restituire più valori: arrivano come tupla

def chiudi(*client): # *client raccoglie tutti gli argomenti passati in una tupla: chiudi(pub, sub) chiude entrambi
    for c in client:
        c.disconnect() # manda DISCONNECT al broker
        c.loop_stop() # ferma il thread di rete del client

In [ ]:
arrivati.clear() # svuoto la lista, se la cella viene rieseguita
sub = iscriviti("test/#")
pub, t0 = pubblica("test/python", "ciao da Python") # i due valori restituiti finiscono in due variabili
time.sleep(0.5) # aspetto mezzo secondo: intanto il messaggio arriva e on_message lo aggiunge alla lista
chiudi(pub, sub)
arrivati

# la callback è registrata in iscriviti(): mqtt_client(..., on_message=on_message) dice a paho quale funzione chiamare all'arrivo di un messaggio

Per la prova e per la chiusura servono gli stessi passi più volte: li mettiamo in una funzione,
`un_messaggio`, che sottoscrive, pubblica, aspetta e restituisce il primo messaggio arrivato con
il suo ritardo in millisecondi, oppure `None` se non arriva nulla.

In [ ]:
def un_messaggio(topic, testo, qos_pub=0, qos_sub=0, filtro="test/#", attesa_s=0.5):
    arrivati.clear()
    sub = iscriviti(filtro, qos_sub)
    pub, t0 = pubblica(topic, testo, qos_pub)
    time.sleep(attesa_s) # aspetto: intanto on_message, sul thread di rete, riempie la lista
    chiudi(pub, sub)
    if not arrivati: # una lista vuota vale False
        print(f"nessun messaggio su '{filtro}' entro {attesa_s} s")
        return None # None: «nessun valore»
    m = dict(arrivati[0]) # copia del primo messaggio arrivato
    t_arrivo = m["t"] # l'istante in cui il messaggio è arrivato, in secondi
    ritardo_s = t_arrivo - t0  # quanto tempo è passato dalla pubblicazione
    m["ritardo_ms"] = round(ritardo_s * 1000, 2) # da secondi a millisecondi, arrotondato a 2 decimali
    del m["t"] #rmmuovo tempo di arrivo
    return m

un_messaggio("test/python", "ciao da Python")

In [ ]:
for i in range(20): # range crea un iterabile con i valori da 0 all'argomento, con passo di default 1
    m = un_messaggio("test/python", "prova") # inviamo un messagio 20 volte
    print(m)

### Prova a cambiare

Si cambiano solo i valori fra le due righe `PARAMETRI`, poi si riesegue la cella.

**Prima di rieseguire: che cosa ti aspetti?** Con topic `prova/python` e filtro `test/#`, il messaggio arriva al notebook? E al terminale 2? Con `FILTRO = "#"`, con quale QoS arriva un messaggio pubblicato a QoS 2 su una sottoscrizione a QoS 0?

In [ ]:
# ── PARAMETRI 
TOPIC = "prova/python"      # il topic su cui pubblico; prova "test/python" e "prova/python"
MESSAGGIO = "ciao di nuovo" # il testo, qualunque stringa
QOS_PUB = 0                 # QoS del pubblicatore: 0, 1 o 2
QOS_SUB = 0                 # QoS del sottoscrittore: 0, 1 o 2
FILTRO = "test/#"           # il filtro del notebook; prova "test/+", "#", "prova/#"
# ───────────────────────────────────────────────────────────────────────────
un_messaggio(TOPIC, MESSAGGIO, QOS_PUB, QOS_SUB, FILTRO) # la funzione definita sopra, con i valori dei PARAMETRI

**Che cosa guardare.** Un topic che il filtro non copre non arriva a nessuno e il broker non lo segnala: il pubblicatore non sa chi riceve. Il campo `qos` del messaggio ricevuto è il minimo fra QoS di pubblicazione e QoS di sottoscrizione. Il ritardo su `localhost` è dell'ordine del millisecondo: su una rete reale lo decide il collegamento.

1. Il pubblicatore non riceve errori quando nessuno è sottoscritto al topic. In un sistema di monitoraggio, come si accorge chi riceve che un nodo ha smesso di pubblicare?
2. Perché il broker consegna a QoS 0 un messaggio pubblicato a QoS 2, se la sottoscrizione è a QoS 0?

## Prova a scrivere

**Calcolare** il ritardo mediano, in millisecondi, fra la pubblicazione di un
messaggio e la sua consegna da parte del broker sul proprio portatile, su 20 messaggi a QoS 0.
Serve la funzione `un_messaggio` della Parte 2: restituisce un dizionario con il campo
`ritardo_ms`, oppure `None` se il messaggio non è arrivato.

In [ ]:
ritardi = []
for i in range(20):
    m = un_messaggio("test/ritardo", f"messaggio {i}")
    if m is not None:
        ritardi.append(m["ritardo_ms"])

print(f"{len(ritardi)} messaggi consegnati su 20; ritardo mediano {np.median(ritardi):.2f} ms")

Su `localhost` il messaggio non esce dal portatile: il ritardo misura solo il lavoro di client e
broker, ed è dell'ordine del millisecondo
